<div align="right"><a href="L34-Burg-Algorithmus.ipynb">Switch to German language 🇩🇪</a></div>

Die folgenden Übungen sind Teil des Buches
* K.-D. Kammeyer, K. Kroschel: "Digitale Signalverarbeitung - Filterung und Spektralanalyse", Springer Vieweg, 2022.  [[KK22]](https://doi.org/10.1007/978-3-663-09805-8)

---

This Jupyter Notebook can be opened in [Google Colab](https://colab.research.google.com/notebooks/intro.ipynb) by clicking on the icon on the right, but this notebook is displayed best when opened locally using [Jupyter Notebook](https://jupyter.org/) (e.g. as part of the [Anaconda](https://www.anaconda.com/) distribution). Please note that for running MATLAB in Notebooks, you have to install the [MATLAB API for Python](https://mathworks.com/help/matlab/matlab-engine-for-python.html) first. The installation process is described [here](https://mathworks.com/help/matlab/matlab_external/install-the-matlab-engine-for-python.html). <a href="https://colab.research.google.com/github/Stefan-Goe/dsp-book/blob/main/matlab/S34-Burg-Algorithm.ipynb"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open Notebook in Google Colab"></a>

---

<!--NAVIGATION-->
<div align="center">

Previous: [S33 Yule-Walker-/Autokorrelationsmethode](./S33-Yule-Walker-und-Autokorrelationsmethode.ipynb) | [Exercise 34](./E34-Burg-Algorithm.ipynb) | [Contents / Overview](./00-MATLAB-Notebooks-Overview-EN.ipynb) | Next: [S35 Comparison of Spectral Estimators](./S35-SpectralEstimators.ipynb)

</div>

---


## Exercise: Burg

The basic idea of the Burg algorithm is to use the balance of the power  
of the forward and backward prediction error  
and to minimize both powers jointly.  
This approach yields an unbiased estimation of the coefficients of the AR($n$)-model  
(in contrast to the autocorrelation method), additionally the analysis filter  
is always a minimum phase filter (in contrast to the autocovariance method).

We’ll start with an (all-pole) system  
$H(z) = \frac{1}{A(z)}$  
with  

$$
A(z) = 1 + 0.3611z^{-1} - 0.1373z^{-2} + 0.2685z^{-3}
- 0.0533z^{-4} - 0.3353z^{-5} - 0.307z^{-6}
$$

of order $m = 6$, which is excited by a stationary, zero mean, white noise process  
$N(k)$ with power $\sigma_N^2 = 1$  
(command: `filter(·)`).

<img src="./pngs/l15sys.eps" style="width:70%;">

---

### a)

Plot the impulse response $h(k)$ of the system in the interval  

$$
0 \le k \le 80
$$

plot the pole-zero plot using the command `zplane(·)`.

---

### b)

Estimate the coefficients of an AR-system AR($n$)

$$
\frac{1}{A_n(z)} = \frac{1}{a_0 + a_1 z^{-1} + \cdots + a_n z^{-n}}
$$

based on $N = 2^{13}$ samples of the process $X(k)$  
for different orders $n = 2,\; 6,\; 32$  
(command: `lburg(·)`).

Compare the different estimated power density spectra among each other  
and with the exact power density spectrum  
$S_{XX}(e^{j\Omega})$.

Why is the estimation for $n = 6$ good?

---

### c)

Repeat exercises a) and b) with a FIR-system

$$
H(z) = 1 - 4z^{-1} + 6z^{-2} - 4z^{-3} + z^{-4}
$$

of order $m = 4$.

Observe the arrangement of the poles of the AR($n$)-model  
to compensate the (4-times) zero of $H(z)$ at $z = 1$.

In [ ]:
# BEGIN SOLUTION
MA = [1]; 
AR = [1.0000 0.3611 -0.1373 0.2685 -0.0533 -0.3353 -0.307];
NFFT=2^10;

% #####  Part a:  AR-System, Transient Behavior #####
% exact ACF (steady state)
[h_arma,t] = impz(MA,AR,80);

% # Graphical Output # %
figure;
stem(t,h_arma); 
grid; 
xlabel('k'); 
ylabel('h(k)');
title(sprintf('Impulse Response of the AR(%d) Model',length(AR)-1));

figure; 
[h1,h2,h3]=zplane(MA,AR); set(h1,'Visible','off');
title(sprintf('Poles of the AR(%d) Model',length(AR)-1)); 
grid;
xlabel('Real Part'); 
ylabel('Imaginary Part');

input('....Press RETURN to continue');

% #####  Part b:  AR-System, PSD #####
% Analysis of the filter order AR(n)
Omega_norm = 0:1/NFFT:1-1/NFFT;
Sxx_arma = abs(fft(MA,NFFT)./fft(AR,NFFT)).^2;
n = [2 length(AR)-1 32];
N = 2^13;
N_vor = 80;
Noise = randn(1,N+N_vor);
x_all = filter(MA,AR,Noise);
x = x_all(N_vor+1:length(x_all));

for k = 1:3
    ar = lburg_algo(x,N,n(k),MA,AR);
    Sxx_ar = abs(ones(1,NFFT)./fft(ar,NFFT)).^2;
    Sxx_ar = Sxx_ar/sum(Sxx_ar).*sum(Sxx_arma);
    % # Graphical Output # %
    figure;
    h1 = plot(Omega_norm, 10*log10(Sxx_arma/max(Sxx_arma)),'r--');  
    hold on;
    h2 = plot(Omega_norm, 10*log10(Sxx_ar/max(Sxx_ar)),'b-');
    ylabel('S_{xx}(e^{j*\Omega}) in dB');
    title(sprintf('PSD from Burg-AR(%d), N=%d', n(k),N)); 
    xlabel('\Omega/2\pi'); 
    grid;
    %legend([h1(1),h2(1)],'exact PSD','estimated PSD',3);
    
    figure; 
    zplane(AR,ar); 
    title(sprintf('Poles of Burg-AR(%d)',n(k)));
    xlabel('Real Part'); 
    ylabel('Imaginary Part'); 
    grid;
end;

input('....Press RETURN to continue');

% #####  Part c:  MA-System #####
% Analysis of the filter order AR(n)
MA = [1 -4 6 -4 1]; 
AR = 1;

% Transient Behavior
[h_arma,t] = impz(MA,AR);

% # Graphical Output # %
figure; 
stem(t,h_arma); 
grid; 
xlabel('k'); 
ylabel('h(k)');
title(sprintf('Impulse Response of the MA(%d) Model',length(MA)-1));

figure; 
zplane(MA,AR);
title(sprintf('Poles of the MA(%d) Model',length(MA)-1));
xlabel('Real Part'); 
ylabel('Imaginary Part'); 
grid;

input('....Press RETURN to continue');

% Analysis of the filter order AR(n), PSD
Omega_norm  = 0:1/NFFT:1-1/NFFT;
Sxx_arma = abs(fft(MA,NFFT)./fft(AR,NFFT)).^2;

n = [2 6 32];
N = 2^13;
N_vor = 5;
noise = randn(1,N+N_vor);
x_all = filter(MA,AR,noise);
x = x_all(N_vor+1:length(x_all));

for k = 1:3
    ar = lburg_algo(x,N,n(k),MA,AR);
    Sxx_ar = abs(ones(1,NFFT)./fft(ar,NFFT)).^2;
    Sxx_ar = Sxx_ar/sum(Sxx_ar).*sum(Sxx_arma);
    
    % # Graphical Output # %
    figure;
    Sxx_ar(find(Sxx_ar==0)) = 1e-10;
    Sxx_arma(find(Sxx_arma==0)) = 1e-10;
    h1 = plot(Omega_norm, 10*log10(Sxx_arma/max(Sxx_arma)),'r--');  
    hold on;
    h2 = plot(Omega_norm, 10*log10(Sxx_ar/max(Sxx_ar)),'b-');
    xlabel('\Omega/2\pi'); 
    ylabel('S_{xx}(e^{j*\Omega}) in dB');
    title(sprintf('PSD from Burg-AR(%d), N=%d', n(k),N)); 
    grid;
    %legend([h1(1),h2(1)],'exact PSD','estimated PSD',3);
    axis([0 1 -70 0]);
    
    figure; 
    zplane(AR,ar); 
    title(sprintf('Poles of Burg-AR(%d)',n(k)));
    xlabel('Real Part'); 
    ylabel('Imaginary Part'); 
    grid;
end;
% #####  EOF  #####
# END SOLUTION

## Solution: Burg

### a)

The following figure shows the impulse response $h(k)$ of the AR(6)-system  
for the interval $0 \le k \le 80$  
as well as the corresponding pole-zero plot  
with poles at  

- $z_{\infty,1} = 0.3444 + i\cdot 0.8315$  
- $z_{\infty,2} = z_{\infty,1}^{*}$  
- $z_{\infty,3} = 0.85$  
- $z_{\infty,4} = -0.91$  
- $z_{\infty,5} = -0.49 + i\cdot 0.4950$  
- $z_{\infty,6} = z_{\infty,5}^{*}$  

<div style="display:flex; justify-content:center; gap:20px;">
  <img src="./pngs/ldsv9a03_a1.png" style="width:45%;">
  <img src="./pngs/ldsv9a03_a2.png" style="width:45%;">
</div>

---

### b)

Procedures for spectrum estimation (by Yule–Walker resp. Burg)  
try to model an AR-filtered white noise process  
(i.e. determine the AR-parameters  
$\hat{a}_{0,n}, \ldots, \hat{a}_{n,n}$),  
so that its statistical properties (the ACF) are nearly  
those of the given (stationary, arbitrary spectrum) process $X(k)$.

This works best if the process is based on AR-filtering and if the  
order of the AR($n$)-model equals the order of the filter.  
The case of filtering with a $m=6$th order AR-system is depicted below.

<div style="display:flex; justify-content:center; gap:20px;">
  <img src="./pngs/ldsv9a03_b11.png" style="width:45%;">
  <img src="./pngs/ldsv9a03_b12.png" style="width:45%;">
</div>

The two figures above depict the exact power density spectra compared to  
power density spectra whose AR-coefficients are estimated with the  
Burg algorithm of order $n = 2, 6, 32$.

The lower pictures represent the corresponding pole positions, whereby  
the poles of $H(z)$ are exceptionally depicted as *circles*.  
All calculations are based on a single sample function of length  
$N = 2^{13} = 8192$ to yield a statistically sufficient estimation.

If the order of the AR-model is underestimated (e.g. $n = 2$), the  
Burg algorithm tries to find the best possible compromise of the pole  
locations. Nevertheless, a significant deviation between the estimated  
and exact power density spectrum exists.

<div style="display:flex; justify-content:center; gap:20px;">
  <img src="./pngs/ldsv9a03_b21.png" style="width:45%;">
  <img src="./pngs/ldsv9a03_b22.png" style="width:45%;">
</div>

If the order of the AR($n$)-model equals the order of the used filter  
$H(z)$, the approximation quality is—as expected—the best.

<div style="display:flex; justify-content:center; gap:20px;">
  <img src="./pngs/ldsv9a03_b31.png" style="width:45%;">
  <img src="./pngs/ldsv9a03_b32.png" style="width:45%;">
</div>

In the case of over-estimation with $n = 32$, all residual coefficients  
are set to nearly zero:

$$
\hat{a}_{n,i} \approx 0 \quad \text{for} \quad i > m
$$

These zeros correspond to $n$ poles placed on a circle near the unit  
circle. Therefore, the corresponding power density spectrum shows a  
more “wavy” course. However, this results only in a slight degradation.

---

### c)

The following graph shows the impulse response $h(k)$ of the MA(4)-system  
for the interval $0 \le k \le 4$.

$H(z)$ is an $m = 4$th order MA-model, so the impulse response has finite  
length $m + 1 = 5$:

$$
h(k) = \{1, -4, 6, -4, 1\}
$$

Therefore, the energy-correlation function  
$r_{HH}^{(E)}(\kappa)$ and thus $r_{XX}(\kappa)$  
is limited to $|\kappa| \le 4$.

This yields the values:

- $r_{XX}(0) = 70$  
- $r_{XX}(\pm 1) = -56$  
- $r_{XX}(\pm 2) = 28$  
- $r_{XX}(\pm 3) = -8$  
- $r_{XX}(\pm 4) = 1$  
- $r_{XX}(\kappa) = 0$ for $|\kappa| \ge 5$

Observe that $H(z)$ has a fourfold zero at $z = 1$ (high-pass).

<div style="display:flex; justify-content:center; gap:20px;">
  <img src="./pngs/ldsv9a03_c1.png" style="width:45%;">
  <img src="./pngs/ldsv9a03_c2.png" style="width:45%;">
</div>

---

### d)

Using the Burg algorithm, we now approximate the MA-model $H(z)$  
with AR-models $1/A_n(z)$ of different orders $n = 2, 6, 32$.

<div style="display:flex; justify-content:center; gap:20px;">
  <img src="./pngs/ldsv9a03_d11.png" style="width:45%;">
  <img src="./pngs/ldsv9a03_d12.png" style="width:45%;">
</div>

<div style="display:flex; justify-content:center; gap:20px;">
  <img src="./pngs/ldsv9a03_d21.png" style="width:45%;">
  <img src="./pngs/ldsv9a03_d22.png" style="width:45%;">
</div>

<div style="display:flex; justify-content:center; gap:20px;">
  <img src="./pngs/ldsv9a03_d31.png" style="width:45%;">
  <img src="./pngs/ldsv9a03_d32.png" style="width:45%;">
</div>

The results show that the Burg algorithm approximates the zeros using a  
constellation of poles near the unit circle.

For low order ($n = 2$), poles are placed as far as possible from the zero  
and arranged on a circle directed toward the zero.  
The surrounding area of the zero is kept free.

From the estimated spectral densities we see that a satisfactory  
approximation is achieved when using a higher-order AR-model.